In [1]:
"""
T6 Milestone 0 — Analysis Notebook

This notebook is the M0 deliverable for the T6 Multi-Objective Vector Scores project.
It demonstrates:
  1. Current baseline behavior (Guide score contract, evaluator aggregation, trainer selection)
  2. Exact code touchpoints and signatures in the OpenTrace codebase
  3. Planned behavior prototype: Pareto front vs weighted selection on deterministic toy candidates

Runs end-to-end WITHOUT API keys.
"""

'\nT6 Milestone 0 — Analysis Notebook\n\nThis notebook is the M0 deliverable for the T6 Multi-Objective Vector Scores project.\nIt demonstrates:\n  1. Current baseline behavior (Guide score contract, evaluator aggregation, trainer selection)\n  2. Exact code touchpoints and signatures in the OpenTrace codebase\n  3. Planned behavior prototype: Pareto front vs weighted selection on deterministic toy candidates\n\nRuns end-to-end WITHOUT API keys.\n'

# T6 Multi-Objective Vector Scores — M0 Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentOpt/OpenTrace/blob/experimental/docs/dev/multi_objective_design_exploration.ipynb)

**Milestone 0 Deliverable** — Analysis + Technical Plan + Interface Spec

This notebook demonstrates:
1. **Current baseline**: How Guide returns scalar scores, how evaluators aggregate, where selection happens
2. **Exact touchpoints**: The specific lines of code in BasicSearch and Beamsearch that perform scalar selection
3. **Planned behavior**: A deterministic prototype showing weighted vs Pareto selection on toy candidates

**Motivation (why score-as-dict):** adding extra metrics into the *feedback dict/text* can help optimizers (OptoPrime/OPRO), but trainers typically only use the scalar score for ranking/UCB and ignore additional feedback structure. To enable Pareto/weighted multi-objective selection at the trainer level, we need vector score (score-as-dict) with backward-compatible scalar reduction.

**No API keys required for M0.** All examples use deterministic dummy data. (From M1 onward, milestone notebooks must validate both StubLLM and real LLM modes.)

---

## How to Validate This Milestone

After running all cells, confirm:
- [ ] Current Guide score contract is printed (`get_feedback → Tuple[float, str]`, `metric → float`)
- [ ] Scalar selection points in BasicSearch and Beamsearch are identified
- [ ] Weighted selection produces different results when weights change
- [ ] Pareto selection returns non-dominated candidates (tradeoff set)
- [ ] Deterministic tie-break produces identical results across repeated runs with same seed

In [2]:
# Setup — no external dependencies beyond numpy
import numpy as np
from typing import Dict, List, Tuple, Optional, Set, Union, Literal
from dataclasses import dataclass, field
import json

print("=" * 70)
print("T6 M0 Analysis — Multi-Objective Vector Scores")
print("=" * 70)

T6 M0 Analysis — Multi-Objective Vector Scores


---
## Part 1: Current Baseline Behavior

### 1.1 Guide Score Contract

The `Guide` base class defines the current score interface:

In [3]:
print("\n" + "=" * 70)
print("PART 1: CURRENT BASELINE BEHAVIOR")
print("=" * 70)

print("""
=== Current Guide Score Contract ===

class Guide:
    def get_feedback(self, query, response, reference=None, **kwargs) -> Tuple[float, str]:
        raise NotImplementedError

    def metric(self, query, response, reference=None, **kwargs) -> float:
        return self.get_feedback(query, response, reference)[0]  # extracts scalar

Key observations:
  • get_feedback() returns Tuple[float, str] — a SCALAR score + feedback string
  • metric() returns float — just extracts the first element
  • LLMJudge (subclass) returns binary 0/1 scores
  • No mechanism to return Dict[str, float] for multiple metrics
""")

# Simulate current behavior
class CurrentGuide:
    """Simulates the current Guide behavior — scalar scores only."""
    def get_feedback(self, query, response, reference=None) -> Tuple[float, str]:
        score = 1.0 if response == reference else 0.0
        feedback = "Correct!" if score == 1.0 else f"Expected '{reference}', got '{response}'"
        return score, feedback

    def metric(self, query, response, reference=None) -> float:
        return self.get_feedback(query, response, reference)[0]

guide = CurrentGuide()
score, feedback = guide.get_feedback("What is 2+2?", "4", "4")
print(f"Example — get_feedback(): score={score} (type={type(score).__name__}), feedback='{feedback}'")
print(f"Example — metric():       {guide.metric('What is 2+2?', '4', '4')} (type={type(guide.metric('What is 2+2?', '4', '4')).__name__})")


PART 1: CURRENT BASELINE BEHAVIOR

=== Current Guide Score Contract ===

class Guide:
    def get_feedback(self, query, response, reference=None, **kwargs) -> Tuple[float, str]:
        raise NotImplementedError

    def metric(self, query, response, reference=None, **kwargs) -> float:
        return self.get_feedback(query, response, reference)[0]  # extracts scalar

Key observations:
  • get_feedback() returns Tuple[float, str] — a SCALAR score + feedback string
  • metric() returns float — just extracts the first element
  • LLMJudge (subclass) returns binary 0/1 scores
  • No mechanism to return Dict[str, float] for multiple metrics

Example — get_feedback(): score=1.0 (type=float), feedback='Correct!'
Example — metric():       1.0 (type=float)


### 1.2 Evaluator Aggregation

In [4]:
print("""
=== Current Evaluator Behavior ===

def evaluate(agent, guide, inputs, infos, ...) -> np.ndarray:
    # For each input: calls guide.metric(input, agent(input), info) → float
    # Returns np.array of shape (N,) or (N, num_samples)
    # Aggregated via np.mean(scores)

Key observations:
  • All scores are numeric scalars
  • Aggregation: np.mean() over all examples
  • No support for Dict[str, float] scores
""")

# Simulate current evaluator
scores_array = np.array([0.9, 0.85, 0.95, 0.7, 0.88])
mean_score = np.mean(scores_array)
print(f"Example — evaluate() returns: {scores_array} (shape={scores_array.shape}, dtype={scores_array.dtype})")
print(f"Example — np.mean(scores):    {mean_score:.4f} (single scalar used for selection)")


=== Current Evaluator Behavior ===

def evaluate(agent, guide, inputs, infos, ...) -> np.ndarray:
    # For each input: calls guide.metric(input, agent(input), info) → float
    # Returns np.array of shape (N,) or (N, num_samples)
    # Aggregated via np.mean(scores)

Key observations:
  • All scores are numeric scalars
  • Aggregation: np.mean() over all examples
  • No support for Dict[str, float] scores

Example — evaluate() returns: [0.9  0.85 0.95 0.7  0.88] (shape=(5,), dtype=float64)
Example — np.mean(scores):    0.8560 (single scalar used for selection)


### 1.3 Selection Points in Trainers

In [5]:
print("""
=== BasicSearchAlgorithm — Selection Logic ===

File: opto/trainer/algorithms/basic_algorithms.py
Method: BasicSearchAlgorithm.optimizer_step()

    def validate():
        scores = evaluate(self.agent, self.validate_guide, ...)
        return np.mean(scores) if all([s is not None for s in scores]) else -np.inf
                 ^^^^^^^^^^^^^^^^
                 Returns: single float

    candidates.append((score, update_dict))       # score is float
    candidates.append((self.current_score, backup_dict))  # include current

    best_score, best_update = max(candidates, key=lambda x: x[0])
                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                               SELECTION: scalar max — single metric only

>>> This is the PRIMARY insertion point for multi-objective selection. <<<
""")

# Simulate current BasicSearch selection
candidates_basic = [
    (0.72, "proposal_A"),
    (0.85, "proposal_B"),
    (0.78, "proposal_C"),
    (0.85, "current_params"),  # tie with proposal_B
]
best_score, best_update = max(candidates_basic, key=lambda x: x[0])
print(f"BasicSearch candidates: {[(s, name) for s, name in candidates_basic]}")
print(f"Selected (scalar max):  score={best_score}, params='{best_update}'")
print(f"Note: Tie between proposal_B and current_params — max() picks first occurrence (proposal_B)")


=== BasicSearchAlgorithm — Selection Logic ===

File: opto/trainer/algorithms/basic_algorithms.py
Method: BasicSearchAlgorithm.optimizer_step()

    def validate():
        scores = evaluate(self.agent, self.validate_guide, ...)
        return np.mean(scores) if all([s is not None for s in scores]) else -np.inf
                 ^^^^^^^^^^^^^^^^
                 Returns: single float

    candidates.append((score, update_dict))       # score is float
    candidates.append((self.current_score, backup_dict))  # include current

    best_score, best_update = max(candidates, key=lambda x: x[0])
                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                               SELECTION: scalar max — single metric only

>>> This is the PRIMARY insertion point for multi-objective selection. <<<

BasicSearch candidates: [(0.72, 'proposal_A'), (0.85, 'proposal_B'), (0.78, 'proposal_C'), (0.85, 'current_params')]
Selected (scalar max):  score=0.85, params='proposal_B'
Note: Tie betw

In [6]:
print("""
=== BeamsearchAlgorithm — Selection Logic ===

File: opto/trainer/algorithms/beamsearch_algorithm.py
Method: BeamsearchAlgorithm.select()

    scored_candidates.append((validation_score, candidate_params))  # float

    sorted_candidates = sorted(scored_candidates, key=lambda x: x[0], reverse=True)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                         SELECTION: scalar sort descending

    selected_candidates = sorted_candidates[:beam_width]   # take top-k
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                          Top-k by scalar score only

>>> This is the SECONDARY insertion point for multi-objective selection. <<<
""")

# Simulate current Beamsearch selection
candidates_beam = [
    (0.72, "candidate_1"),
    (0.91, "candidate_2"),
    (0.85, "candidate_3"),
    (0.91, "candidate_4"),  # tie with candidate_2
    (0.78, "candidate_5"),
]
beam_width = 3
sorted_candidates = sorted(candidates_beam, key=lambda x: x[0], reverse=True)
selected = sorted_candidates[:beam_width]
print(f"Beamsearch candidates: {[(s, name) for s, name in candidates_beam]}")
print(f"Selected (top-{beam_width} by scalar): {[(s, name) for s, name in selected]}")
print(f"Note: Tie between candidate_2 and candidate_4 — sorted() preserves input order (stable sort)")


=== BeamsearchAlgorithm — Selection Logic ===

File: opto/trainer/algorithms/beamsearch_algorithm.py
Method: BeamsearchAlgorithm.select()

    scored_candidates.append((validation_score, candidate_params))  # float

    sorted_candidates = sorted(scored_candidates, key=lambda x: x[0], reverse=True)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                         SELECTION: scalar sort descending

    selected_candidates = sorted_candidates[:beam_width]   # take top-k
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                          Top-k by scalar score only

>>> This is the SECONDARY insertion point for multi-objective selection. <<<

Beamsearch candidates: [(0.72, 'candidate_1'), (0.91, 'candidate_2'), (0.85, 'candidate_3'), (0.91, 'candidate_4'), (0.78, 'candidate_5')]
Selected (top-3 by scalar): [(0.91, 'candidate_2'), (0.91, 'candidate_4'), (0.85, 'candidate_3')]
Note: Tie between candidate_2 and candidate_4 — sorted(

### 1.4 Summary: What's Missing


In [ ]:
print("""
=== Summary: Current Limitations ===

0. Extra metrics in feedback are not usable by trainers today.
   Trainers typically rank/UCB using only the scalar score, and do not inspect feedback structure.

1. Guide.metric() → float only (and stays float BY DESIGN)
   metric() will NOT be widened to return dicts.
   Dict scores flow through the NEW get_score_dict() path instead.

2. evaluate() → np.array of floats
   Cannot aggregate per-metric means across examples.
   New evaluate_vector() will handle dict aggregation separately.

3. BasicSearch: max(candidates, key=scalar)
   Cannot do weighted multi-metric selection or Pareto ranking

4. Beamsearch: sorted(candidates, key=scalar)[:k]
   Cannot select top-k by Pareto dominance

5. No ObjectiveConfig
   No way to declare minimize metrics, weights, or selection mode

>>> All of the above will be addressed in M1-M2 without breaking existing behavior. <<<
>>> Training loop (guide.__call__ → float) is NEVER modified.                      <<<
""")

---
## Part 2: Planned Behavior — Prototype

The following cells implement the **planned multi-objective selection** as pure functions.
This is a standalone prototype (no OpenTrace dependency) demonstrating the exact behavior
that `opto/trainer/objectives.py` will provide.

### 2.1 ObjectiveConfig

In [ ]:
print("\n" + "=" * 70)
print("PART 2: PLANNED BEHAVIOR — PROTOTYPE")
print("=" * 70)

@dataclass(frozen=True)
class ObjectiveConfig:
    """Configuration for multi-objective candidate selection."""
    mode: str = "scalar"  # "scalar", "weighted", "pareto"
    weights: Dict[str, float] = field(default_factory=dict)
    minimize: frozenset = field(default_factory=frozenset)
    missing_value: float = float("-inf")
    pareto_metrics: Optional[Tuple[str, ...]] = None
    tie_break: str = "weighted"  # "weighted", "lexicographic", "random_seeded"
    seed: int = 0

    def __post_init__(self):
        # Convert set → frozenset for true immutability + hashability
        if isinstance(self.minimize, set):
            object.__setattr__(self, 'minimize', frozenset(self.minimize))
        # Validate weights are non-negative
        for k, v in self.weights.items():
            if v < 0:
                raise ValueError(f"Weight for '{k}' must be non-negative, got {v}")
        # Validate pareto_metrics
        if self.pareto_metrics is not None and len(self.pareto_metrics) == 0:
            raise ValueError("pareto_metrics must be None (auto) or non-empty tuple")

print("ObjectiveConfig defined with modes: scalar | weighted | pareto")
print(f"Default config: {ObjectiveConfig()}")

# Verify set → frozenset auto-conversion
config_with_set = ObjectiveConfig(minimize={"latency_s"})
print(f"minimize=set auto-converts: type={type(config_with_set.minimize).__name__}, value={config_with_set.minimize}")

### 2.2 Core Utility Functions

In [ ]:
# --- Score type aliases ---
ScoreLike = Union[int, float, bool, Dict[str, float]]


def normalize_score(score: ScoreLike) -> Dict[str, float]:
    """Convert any score to dict form.
    
    - int/float/bool → {"score": float(value)}
    - Dict[str, float] → returned as-is (validated)
    
    Handles int (LLMJudge returns 0/1) and bool (test guides) explicitly.
    """
    if isinstance(score, (bool, int, float)):
        # bool check must come before int since bool is subclass of int
        val = float(score)
        if not np.isfinite(val):
            raise ValueError(f"Score must be finite, got {score}")
        return {"score": val}
    elif isinstance(score, dict):
        if len(score) == 0:
            raise ValueError("Score dict must not be empty")
        for k, v in score.items():
            if not isinstance(v, (int, float)) or not np.isfinite(float(v)):
                raise ValueError(f"Score dict value for '{k}' must be finite float, got {v}")
        return {k: float(v) for k, v in score.items()}
    else:
        raise TypeError(f"Score must be int, float, bool, or Dict[str, float], got {type(score).__name__}")


def apply_minimize(score_dict: Dict[str, float], minimize: set) -> Dict[str, float]:
    """Negate values for minimize metrics (higher-is-better normalization)."""
    return {
        k: -v if k in minimize else v
        for k, v in score_dict.items()
    }


def weighted_scalarize(score_dict: Dict[str, float], weights: Dict[str, float],
                       missing_value: float = float("-inf")) -> float:
    """Compute weighted sum. Empty weights → equal weight 1.0."""
    if not weights:
        weights = {k: 1.0 for k in score_dict}
    total = 0.0
    for metric, weight in weights.items():
        value = score_dict.get(metric, missing_value)
        total += weight * value
    return total


def dominates(a: Dict[str, float], b: Dict[str, float],
              metrics: Optional[Tuple[str, ...]] = None) -> bool:
    """Check if candidate 'a' Pareto-dominates candidate 'b'.
    
    a dominates b iff:
      - a[m] >= b[m] for ALL metrics m, AND
      - a[m] > b[m] for AT LEAST ONE metric m
    """
    if metrics is None:
        metrics = tuple(sorted(set(a.keys()) | set(b.keys())))
    
    at_least_one_better = False
    for m in metrics:
        va = a.get(m, float("-inf"))
        vb = b.get(m, float("-inf"))
        if va < vb:
            return False  # a is worse on this metric
        if va > vb:
            at_least_one_better = True
    return at_least_one_better


def pareto_rank(candidates: List[Dict[str, float]],
                metrics: Optional[Tuple[str, ...]] = None) -> List[int]:
    """Assign Pareto rank (0 = non-dominated front)."""
    n = len(candidates)
    ranks = [0] * n
    assigned = [False] * n
    current_rank = 0

    remaining = set(range(n))
    while remaining:
        # Find non-dominated set among remaining
        front = []
        for i in remaining:
            dominated = False
            for j in remaining:
                if i != j and dominates(candidates[j], candidates[i], metrics):
                    dominated = True
                    break
            if not dominated:
                front.append(i)

        for i in front:
            ranks[i] = current_rank
            remaining.remove(i)
        current_rank += 1

    return ranks


def select_best(candidates: List[Tuple[ScoreLike, any]],
                config: Optional[ObjectiveConfig] = None) -> int:
    """Select the single best candidate index."""
    if config is None or config.mode == "scalar":
        # Backward-compatible: scalar max
        scores = []
        for score, _ in candidates:
            if isinstance(score, dict):
                scores.append(np.mean(list(score.values())))
            else:
                scores.append(float(score))
        return int(np.argmax(scores))

    # Normalize all scores to dict
    score_dicts = [normalize_score(s) for s, _ in candidates]

    # Apply minimize
    score_dicts = [apply_minimize(sd, config.minimize) for sd in score_dicts]

    if config.mode == "weighted":
        weighted_scores = [weighted_scalarize(sd, config.weights, config.missing_value) for sd in score_dicts]
        return int(np.argmax(weighted_scores))

    elif config.mode == "pareto":
        ranks = pareto_rank(score_dicts, config.pareto_metrics)
        # Get indices of rank-0 (Pareto front)
        front_indices = [i for i, r in enumerate(ranks) if r == 0]

        if len(front_indices) == 1:
            return front_indices[0]

        # Tie-break among front
        if config.tie_break == "weighted":
            front_scores = [weighted_scalarize(score_dicts[i], config.weights, config.missing_value)
                           for i in front_indices]
            return front_indices[int(np.argmax(front_scores))]
        elif config.tie_break == "lexicographic":
            metrics = sorted(score_dicts[front_indices[0]].keys())
            def lex_key(idx):
                return tuple(score_dicts[idx].get(m, config.missing_value) for m in metrics)
            return max(front_indices, key=lex_key)
        elif config.tie_break == "random_seeded":
            rng = np.random.RandomState(config.seed)
            return front_indices[rng.randint(len(front_indices))]

    raise ValueError(f"Unknown mode: {config.mode}")


def select_top_k(candidates: List[Tuple[ScoreLike, any]],
                 config: Optional[ObjectiveConfig] = None,
                 k: int = 1) -> List[int]:
    """Select the top-k candidate indices."""
    if config is None or config.mode == "scalar":
        scores = []
        for score, _ in candidates:
            if isinstance(score, dict):
                scores.append(np.mean(list(score.values())))
            else:
                scores.append(float(score))
        return list(np.argsort(scores)[::-1][:k])

    score_dicts = [normalize_score(s) for s, _ in candidates]
    score_dicts = [apply_minimize(sd, config.minimize) for sd in score_dicts]

    if config.mode == "weighted":
        weighted_scores = [weighted_scalarize(sd, config.weights, config.missing_value) for sd in score_dicts]
        return list(np.argsort(weighted_scores)[::-1][:k])

    elif config.mode == "pareto":
        ranks = pareto_rank(score_dicts, config.pareto_metrics)
        # Collect by rank, then tie-break within each rank
        result = []
        max_rank = max(ranks)
        for rank in range(max_rank + 1):
            rank_indices = [i for i, r in enumerate(ranks) if r == rank]
            # Sort within rank by tie-break
            if config.tie_break == "weighted":
                rank_indices.sort(
                    key=lambda i: weighted_scalarize(score_dicts[i], config.weights, config.missing_value),
                    reverse=True
                )
            elif config.tie_break == "lexicographic":
                metrics = sorted(score_dicts[rank_indices[0]].keys()) if rank_indices else []
                rank_indices.sort(
                    key=lambda i: tuple(score_dicts[i].get(m, config.missing_value) for m in metrics),
                    reverse=True
                )
            elif config.tie_break == "random_seeded":
                rng = np.random.RandomState(config.seed + rank)
                rng.shuffle(rank_indices)
            result.extend(rank_indices)
            if len(result) >= k:
                break
        return result[:k]

    raise ValueError(f"Unknown mode: {config.mode}")


print("Core utility functions defined:")
print("  \u2022 normalize_score()  — handles float, int, bool, and dict")
print("  \u2022 apply_minimize()")
print("  \u2022 weighted_scalarize()")
print("  \u2022 dominates()")
print("  \u2022 pareto_rank()")
print("  \u2022 select_best()")
print("  \u2022 select_top_k()")

### 2.3 Validation: normalize_score()

In [ ]:
print("\n--- normalize_score() examples ---")
print(f"  normalize_score(0.85)                     = {normalize_score(0.85)}")
print(f"  normalize_score({{'acc': 0.9, 'lat': 50}}) = {normalize_score({'acc': 0.9, 'lat': 50})}")

# int and bool edge cases (LLMJudge returns int 0/1, test guides return bool)
print(f"\n  --- int / bool edge cases ---")
print(f"  normalize_score(1)       = {normalize_score(1)}       # LLMJudge returns int 0/1")
print(f"  normalize_score(0)       = {normalize_score(0)}       # LLMJudge incorrect → int 0")
print(f"  normalize_score(True)    = {normalize_score(True)}    # test guide correct → bool")
print(f"  normalize_score(False)   = {normalize_score(False)}    # test guide incorrect → bool")

# Error edge cases
print(f"\n  --- Error edge cases ---")
try:
    normalize_score({})
except ValueError as e:
    print(f"  normalize_score({{}})                       → ValueError: {e}")

try:
    normalize_score("bad")
except TypeError as e:
    print(f"  normalize_score('bad')                    → TypeError: {e}")

### 2.4 Validation: apply_minimize() + weighted_scalarize()

In [11]:
print("\n--- apply_minimize() examples ---")
score = {"accuracy": 0.9, "latency_ms": 120.0, "cost": 0.05}
minimized = apply_minimize(score, minimize={"latency_ms", "cost"})
print(f"  Original:  {score}")
print(f"  Minimize:  {{'latency_ms', 'cost'}}")
print(f"  Result:    {minimized}")
print(f"  (latency and cost negated → higher-is-better)")

print("\n--- weighted_scalarize() examples ---")
weights = {"accuracy": 0.6, "latency_ms": 0.3, "cost": 0.1}
ws = weighted_scalarize(minimized, weights)
print(f"  Score (normalized): {minimized}")
print(f"  Weights:            {weights}")
print(f"  Weighted sum:       {ws:.4f}")
print(f"  = 0.6*0.9 + 0.3*(-120.0) + 0.1*(-0.05) = {0.6*0.9 + 0.3*(-120.0) + 0.1*(-0.05):.4f}")


--- apply_minimize() examples ---
  Original:  {'accuracy': 0.9, 'latency_ms': 120.0, 'cost': 0.05}
  Minimize:  {'latency_ms', 'cost'}
  Result:    {'accuracy': 0.9, 'latency_ms': -120.0, 'cost': -0.05}
  (latency and cost negated → higher-is-better)

--- weighted_scalarize() examples ---
  Score (normalized): {'accuracy': 0.9, 'latency_ms': -120.0, 'cost': -0.05}
  Weights:            {'accuracy': 0.6, 'latency_ms': 0.3, 'cost': 0.1}
  Weighted sum:       -35.4650
  = 0.6*0.9 + 0.3*(-120.0) + 0.1*(-0.05) = -35.4650


### 2.5 Demonstration: Weighted vs Pareto Selection

We create 6 candidates with realistic multi-metric scores to show how weighted and Pareto selection differ.

In [12]:
print("\n" + "=" * 70)
print("DEMONSTRATION: WEIGHTED vs PARETO SELECTION")
print("=" * 70)

# 6 candidates with accuracy (higher=better) and latency_s (lower=better)
# Using latency_s (seconds, 0-1 scale) so metrics are comparable and weight changes matter
candidates = [
    ({"accuracy": 0.95, "latency_s": 0.200}, "candidate_A"),  # high accuracy, high latency
    ({"accuracy": 0.70, "latency_s": 0.030}, "candidate_B"),  # low accuracy, low latency
    ({"accuracy": 0.88, "latency_s": 0.080}, "candidate_C"),  # balanced
    ({"accuracy": 0.92, "latency_s": 0.150}, "candidate_D"),  # good accuracy, moderate latency
    ({"accuracy": 0.60, "latency_s": 0.020}, "candidate_E"),  # lowest accuracy, fastest
    ({"accuracy": 0.85, "latency_s": 0.085}, "candidate_F"),  # similar to C
]

print("\nCandidate Scores:")
print(f"  {'Name':<15} {'Accuracy':>10} {'Latency (s)':>12}")
print(f"  {'-'*15} {'-'*10} {'-'*12}")
for score, name in candidates:
    print(f"  {name:<15} {score['accuracy']:>10.2f} {score['latency_s']:>12.3f}")

# --- Scalar mode (baseline) ---
print("\n--- Mode: SCALAR (baseline) ---")
config_scalar = ObjectiveConfig(mode="scalar")
best_idx = select_best(candidates, config_scalar)
print(f"  Selection: mean of dict values → max")
print(f"  Winner: {candidates[best_idx][1]} (index {best_idx})")
print(f"  Score: {candidates[best_idx][0]}")
print(f"  Note: This is the CURRENT behavior — treats multi-metric as mean scalar.")

# --- Weighted mode: accuracy-heavy ---
print("\n--- Mode: WEIGHTED (accuracy-heavy) ---")
config_weighted_acc = ObjectiveConfig(
    mode="weighted",
    weights={"accuracy": 0.8, "latency_s": 0.2},
    minimize=frozenset({"latency_s"})
)
best_idx = select_best(candidates, config_weighted_acc)
print(f"  Weights: accuracy=0.8, latency_s=0.2 (minimized)")
print(f"  Winner: {candidates[best_idx][1]} (index {best_idx})")
score_dict = apply_minimize(candidates[best_idx][0], config_weighted_acc.minimize)
ws = weighted_scalarize(score_dict, config_weighted_acc.weights)
print(f"  Weighted score: {ws:.4f}")

# --- Weighted mode: latency-heavy ---
print("\n--- Mode: WEIGHTED (latency-heavy) ---")
config_weighted_lat = ObjectiveConfig(
    mode="weighted",
    weights={"accuracy": 0.2, "latency_s": 0.8},
    minimize=frozenset({"latency_s"})
)
best_idx_lat = select_best(candidates, config_weighted_lat)
print(f"  Weights: accuracy=0.2, latency_s=0.8 (minimized)")
print(f"  Winner: {candidates[best_idx_lat][1]} (index {best_idx_lat})")
score_dict_lat = apply_minimize(candidates[best_idx_lat][0], config_weighted_lat.minimize)
ws_lat = weighted_scalarize(score_dict_lat, config_weighted_lat.weights)
print(f"  Weighted score: {ws_lat:.4f}")

print(f"\n  >>> Changing weights changes the winner!")
print(f"  >>> Accuracy-heavy → {candidates[best_idx][1]}, Latency-heavy → {candidates[best_idx_lat][1]}")

# --- Pareto mode ---
print("\n--- Mode: PARETO ---")
config_pareto = ObjectiveConfig(
    mode="pareto",
    weights={"accuracy": 0.5, "latency_s": 0.5},  # used for tie-breaking
    minimize=frozenset({"latency_s"}),
    tie_break="weighted",
    seed=42
)

# Show full Pareto ranking
score_dicts_norm = [apply_minimize(normalize_score(s), config_pareto.minimize) for s, _ in candidates]
ranks = pareto_rank(score_dicts_norm)

print(f"\n  Pareto Ranking (after minimize normalization):")
print(f"  {'Name':<15} {'Accuracy':>10} {'Neg Latency':>12} {'Pareto Rank':>12}")
print(f"  {'-'*15} {'-'*10} {'-'*12} {'-'*12}")
for i, ((score, name), rank) in enumerate(zip(candidates, ranks)):
    nd = score_dicts_norm[i]
    print(f"  {name:<15} {nd['accuracy']:>10.2f} {nd['latency_s']:>12.3f} {rank:>12}")

front_indices = [i for i, r in enumerate(ranks) if r == 0]
print(f"\n  Pareto Front (Rank 0): {[candidates[i][1] for i in front_indices]}")
print(f"  These candidates represent TRADEOFFS — none is dominated by another.")

best_idx_pareto = select_best(candidates, config_pareto)
print(f"\n  After tie-break (weighted, weights={{acc: 0.5, lat: 0.5}}):")
print(f"  Winner: {candidates[best_idx_pareto][1]} (index {best_idx_pareto})")

# --- Top-k selection (Beamsearch simulation) ---
print("\n--- Mode: PARETO (top-k for Beamsearch, k=3) ---")
top_k_indices = select_top_k(candidates, config_pareto, k=3)
print(f"  Selected top-3:")
for rank_pos, idx in enumerate(top_k_indices):
    r = ranks[idx]
    print(f"    #{rank_pos+1}: {candidates[idx][1]} (Pareto rank {r}, scores: {candidates[idx][0]})")


DEMONSTRATION: WEIGHTED vs PARETO SELECTION

Candidate Scores:
  Name              Accuracy  Latency (s)
  --------------- ---------- ------------
  candidate_A           0.95        0.200
  candidate_B           0.70        0.030
  candidate_C           0.88        0.080
  candidate_D           0.92        0.150
  candidate_E           0.60        0.020
  candidate_F           0.85        0.085

--- Mode: SCALAR (baseline) ---
  Selection: mean of dict values → max
  Winner: candidate_A (index 0)
  Score: {'accuracy': 0.95, 'latency_s': 0.2}
  Note: This is the CURRENT behavior — treats multi-metric as mean scalar.

--- Mode: WEIGHTED (accuracy-heavy) ---
  Weights: accuracy=0.8, latency_s=0.2 (minimized)
  Winner: candidate_A (index 0)
  Weighted score: 0.7200

--- Mode: WEIGHTED (latency-heavy) ---
  Weights: accuracy=0.2, latency_s=0.8 (minimized)
  Winner: candidate_B (index 1)
  Weighted score: 0.1160

  >>> Changing weights changes the winner!
  >>> Accuracy-heavy → candidate_A

### 2.6 Deterministic Tie-Break Validation

In [13]:
print("\n" + "=" * 70)
print("DETERMINISTIC TIE-BREAK VALIDATION")
print("=" * 70)

# Run selection 10 times with same seed — must produce identical results
print("\n--- Repeated runs with seed=42 ---")
results = []
for run in range(10):
    config = ObjectiveConfig(
        mode="pareto",
        weights={"accuracy": 0.5, "latency_s": 0.5},
        minimize=frozenset({"latency_s"}),
        tie_break="weighted",
        seed=42
    )
    idx = select_best(candidates, config)
    results.append(idx)

all_same = len(set(results)) == 1
print(f"  10 runs with seed=42: indices = {results}")
print(f"  All identical: {all_same} ✓" if all_same else f"  NOT identical: FAIL ✗")

# Different seed should potentially give different tie-break (if random_seeded)
print("\n--- Different seeds with random_seeded tie-break ---")
for seed in [0, 1, 2, 42, 99]:
    config = ObjectiveConfig(
        mode="pareto",
        weights={"accuracy": 0.5, "latency_s": 0.5},
        minimize=frozenset({"latency_s"}),
        tie_break="random_seeded",
        seed=seed
    )
    idx = select_best(candidates, config)
    print(f"  seed={seed:>2}: winner = {candidates[idx][1]} (index {idx})")

# Verify same seed is deterministic for random_seeded too
print("\n--- Determinism check for random_seeded (seed=42, 10 runs) ---")
results_random = []
for _ in range(10):
    config = ObjectiveConfig(
        mode="pareto",
        weights={"accuracy": 0.5, "latency_s": 0.5},
        minimize=frozenset({"latency_s"}),
        tie_break="random_seeded",
        seed=42
    )
    idx = select_best(candidates, config)
    results_random.append(idx)
all_same_random = len(set(results_random)) == 1
print(f"  10 runs: indices = {results_random}")
print(f"  All identical: {all_same_random} ✓" if all_same_random else f"  NOT identical: FAIL ✗")


DETERMINISTIC TIE-BREAK VALIDATION

--- Repeated runs with seed=42 ---
  10 runs with seed=42: indices = [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
  All identical: True ✓

--- Different seeds with random_seeded tie-break ---
  seed= 0: winner = candidate_E (index 4)
  seed= 1: winner = candidate_D (index 3)
  seed= 2: winner = candidate_A (index 0)
  seed=42: winner = candidate_D (index 3)
  seed=99: winner = candidate_B (index 1)

--- Determinism check for random_seeded (seed=42, 10 runs) ---
  10 runs: indices = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
  All identical: True ✓


### 2.7 Edge Cases

In [ ]:
print("\n" + "=" * 70)
print("EDGE CASES")
print("=" * 70)

# Single-metric dict
print("\n--- Single-metric dict with Pareto mode ---")
single_metric_candidates = [
    ({"accuracy": 0.9}, "A"),
    ({"accuracy": 0.8}, "B"),
    ({"accuracy": 0.95}, "C"),
]
config_single = ObjectiveConfig(mode="pareto", tie_break="weighted")
best = select_best(single_metric_candidates, config_single)
print(f"  Candidates: {[s for s, _ in single_metric_candidates]}")
print(f"  Winner: {single_metric_candidates[best][1]} (index {best})")
print(f"  Note: Pareto with 1 metric degenerates to scalar max — expected behavior.")

# Mixed float and dict
print("\n--- Backward compat: float scores with ObjectiveConfig ---")
float_candidates = [
    (0.85, "A"),
    (0.92, "B"),
    (0.78, "C"),
]
config_float = ObjectiveConfig(mode="weighted", weights={"score": 1.0})
best_float = select_best(float_candidates, config_float)
print(f"  Float candidates: {[s for s, _ in float_candidates]}")
print(f"  Winner: {float_candidates[best_float][1]} (score={float_candidates[best_float][0]})")
print(f"  Note: Floats normalized to {{'score': val}} — backward-compatible.")

# None config (pure backward compatibility)
print("\n--- None config (current behavior) ---")
best_none = select_best(float_candidates, None)
print(f"  config=None → scalar max → {float_candidates[best_none][1]} (score={float_candidates[best_none][0]})")
print(f"  Identical to current max(candidates, key=lambda x: x[0])")

# Negative weight validation
print("\n--- Negative weight validation ---")
try:
    ObjectiveConfig(weights={"accuracy": 0.8, "latency_s": -0.2})
except ValueError as e:
    print(f"  ObjectiveConfig(weights={{..., 'latency_s': -0.2}}) → ValueError: {e}")
    print(f"  Note: Use minimize={{'latency_s'}} instead of negative weights.")

# Empty pareto_metrics validation
print("\n--- Empty pareto_metrics validation ---")
try:
    ObjectiveConfig(pareto_metrics=())
except ValueError as e:
    print(f"  ObjectiveConfig(pareto_metrics=()) → ValueError: {e}")
    print(f"  Note: Use None (auto-detect) or a non-empty tuple of metric names.")

### 2.8 Visual Summary: Selection Behavior Comparison

In [15]:
print("\n" + "=" * 70)
print("SELECTION BEHAVIOR COMPARISON")
print("=" * 70)

print(f"\n  {'Mode':<25} {'Winner':<15} {'Reasoning'}")
print(f"  {'-'*25} {'-'*15} {'-'*50}")

modes = [
    ("scalar (baseline)", config_scalar),
    ("weighted (acc=0.8)", config_weighted_acc),
    ("weighted (lat=0.8)", config_weighted_lat),
    ("pareto (tie=weighted)", config_pareto),
]

for mode_name, config in modes:
    idx = select_best(candidates, config)
    name = candidates[idx][1]
    if config.mode == "scalar":
        reason = "mean of dict values → max"
    elif config.mode == "weighted":
        reason = f"weighted sum with {dict(config.weights)}"
    elif config.mode == "pareto":
        reason = f"rank-0 front, tie-break={config.tie_break}"
    print(f"  {mode_name:<25} {name:<15} {reason}")

print(f"\n  >>> Different modes select different candidates from the SAME pool.")
print(f"  >>> This is exactly the behavior objectives.py will provide to trainers.")


SELECTION BEHAVIOR COMPARISON

  Mode                      Winner          Reasoning
  ------------------------- --------------- --------------------------------------------------
  scalar (baseline)         candidate_A     mean of dict values → max
  weighted (acc=0.8)        candidate_A     weighted sum with {'accuracy': 0.8, 'latency_s': 0.2}
  weighted (lat=0.8)        candidate_B     weighted sum with {'accuracy': 0.2, 'latency_s': 0.8}
  pareto (tie=weighted)     candidate_C     rank-0 front, tie-break=weighted

  >>> Different modes select different candidates from the SAME pool.
  >>> This is exactly the behavior objectives.py will provide to trainers.


---
## Part 3: Architecture Summary

### Two separate data paths (by design)

The training loop and selection path are **intentionally separate**. `guide.__call__()` / `get_feedback()` return type is NOT widened — the training loop always receives `float`.

```
TRAINING LOOP (unchanged):
  guide(x, target.data, info) → (float, str)
      │
      └── score (float) → np.mean(scores)  → optimizer backward
          Always float. Never dict. Training loop is completely safe.

SELECTION PATH (new):
  guide.get_score_dict(query, response, reference) → Dict[str, float]
      │
      ▼
  evaluate_vector() → List[Dict[str, float]]   (one dict per example)
      │
      ▼
  aggregate_vector_scores() → Dict[str, float]  (mean per metric)
      │
      ▼
  objectives.py (select_best / select_top_k)
      │
      ├── mode="scalar"   → max(mean_scores)           ← unchanged
      ├── mode="weighted"  → max(weighted_scalarize())  ← new
      └── mode="pareto"    → pareto_rank() + tie-break  ← new
```

**Key safety invariant:** `metric()` always returns `float`. If a guide's `get_feedback()` returns a dict as the score, `metric()` collapses it via `mean(values)`. Dict scores are only accessible through `get_score_dict()`.

### Files to create/modify

| Action | File | Milestone |
|--------|------|-----------|
| CREATE | `opto/trainer/objectives.py` | M1 |
| MODIFY | `opto/trainer/guide.py` — add `get_score_dict()`, update `metric()` to collapse dicts to float | M1 |
| MODIFY | `opto/trainer/evaluators.py` — add `evaluate_vector()`, `aggregate_vector_scores()` | M1 |
| MODIFY | `basic_algorithms.py` | M1-M2 |
| MODIFY | `beamsearch_algorithm.py` | M2 |
| OPTIONAL | `priority_search.py` | M2 |

In [16]:
print("\n" + "=" * 70)
print("M0 ANALYSIS COMPLETE")
print("=" * 70)
print("""
Deliverables verified:
  ✓ Current Guide score contract documented (Tuple[float, str])
  ✓ Scalar selection points identified (BasicSearch max, Beamsearch sorted[:k])
  ✓ Weighted selection produces different results with different weights
  ✓ Pareto selection returns non-dominated tradeoff set
  ✓ Deterministic tie-break verified (same seed → same result, 10 runs)
  ✓ Edge cases validated (empty dict, single metric, float compat, None config)
  ✓ Architecture summary with file list and data flow

See docs/T6_technical_plan.md for the complete refined technical plan.
""")


M0 ANALYSIS COMPLETE

Deliverables verified:
  ✓ Current Guide score contract documented (Tuple[float, str])
  ✓ Scalar selection points identified (BasicSearch max, Beamsearch sorted[:k])
  ✓ Weighted selection produces different results with different weights
  ✓ Pareto selection returns non-dominated tradeoff set
  ✓ Deterministic tie-break verified (same seed → same result, 10 runs)
  ✓ Edge cases validated (empty dict, single metric, float compat, None config)
  ✓ Architecture summary with file list and data flow

See docs/T6_technical_plan.md for the complete refined technical plan.

